In [ ]:
########################################################################
# Fluid Simulation Example - Update Variables
#
# Use Visor to visualize a fluid simulation, and update the
# variables in Visor as the simulation progresses.
#
# Dependency: This example requires the Warp library, which is not
# included in the Visor package.
# > pip install warp-lang[examples]
#########################################################################
import asyncio

import numpy as np
import warp as wp
from IPython.display import IFrame
from vtk import vtkHDFWriter, vtkPlaneSource, vtkPolyData
from vtk.util.numpy_support import numpy_to_vtk
from warp.examples.core import example_fluid
from warp.examples.core.example_fluid import Example

from ansys.visor.viewer import Metadata, Visor

wp.ScopedTimer.enabled = False

w = example_fluid.grid_width
h = example_fluid.grid_height
VTKHDF_FILE_PATH = "warp_fluid_sim.vtkhdf"

In [ ]:
########################################################################
# Sim data + VTK dataset configuration
########################################################################

def point_arrays(sim):
    """ Return a dict of the fluid fields from the warp simulation, ordered for VTK point arrays."""
    density = sim.rho0.numpy().T.ravel()
    pressure = sim.p0.numpy().T.ravel()
    velocity = sim.u0.numpy().transpose(1, 0, 2).reshape(-1, 2)
    velocity3 = np.column_stack([velocity, np.zeros(len(velocity), np.float32)])
    return {"density": density, "pressure": pressure, "velocity": velocity3}

def create_grid():
    """Create a quad mesh, of size w x h with one point per fluid cell."""
    plane = vtkPlaneSource()
    plane.SetOrigin(0, 0, 0)
    plane.SetPoint1(w-1, 0, 0)
    plane.SetPoint2(0, h-1, 0)
    plane.SetResolution(w-1, h-1)
    plane.Update()
    polydata = vtkPolyData()
    polydata.DeepCopy(plane.GetOutput())
    # Remove the default arrays so we only have the simulation's variables
    for name in ("Normals", "TextureCoordinates"):
        polydata.GetPointData().RemoveArray(name)
    return polydata

def add_arrays_to_dataset(dataset, arrays):
    """Add point arrays on a VTK dataset"""
    for name, values in arrays.items():
        # Create a vtkDataArray from the numpy array
        array = numpy_to_vtk(np.ascontiguousarray(values, dtype=np.float32), deep=True)
        array.SetName(name)
        dataset.GetPointData().AddArray(array)

def write_vtkhdf(dataset, file_path):
    """Writes a VTK dataset to disk as VTKHDF."""
    writer = vtkHDFWriter()
    writer.SetInputData(dataset)
    writer.SetFileName(file_path)
    if not writer.Write():
        raise RuntimeError(f"vtkHDFWriter failed for {file_path}")

def create_dataset_and_write_to_disk(sim):
    """Creates a VTK dataset, adds arrays from the sim, and writes the dataset to disk."""
    polydata = create_grid()
    arrays = point_arrays(sim)
    add_arrays_to_dataset(polydata, arrays)
    write_vtkhdf(polydata, VTKHDF_FILE_PATH)

########################################################################
# Visor configuration
########################################################################

def set_up_metadata(variable_id="POINT::velocity::3"):
    """Set up metadata for the dataset in Visor.
    
    This is optional; here, it automatically colors by the specified variable
    instead of having to set it manually in the browser.

    Variables available here:
       POINT::density::1
       POINT::pressure::1
       POINT::velocity::3
    """
    metadata = Metadata(
        name="fluid_simulation",
        unit="",
        state={
            "parts": {
                "fluid_simulation": {
                    "color_by": variable_id,     # POINT/CELL :: name :: num_components
                    "color_by_component": -1,    # -1 is magnitude
                }
            }
        }
    )
    return metadata

def configure_visor(port, variable_to_display, mode="CLIENT"):
    """Instantiate VISOR, start with the VTK dataset file path, and get the dataset's part variables."""
    visor = Visor(rendering_mode=mode, url=f"http://localhost:{port}")
    metadata = set_up_metadata(variable_to_display)
    dataset_id = visor.start(input=VTKHDF_FILE_PATH, metadata=metadata)
    part_variables = visor.list_variables(dataset_id)[0]
    return visor, dataset_id, part_variables

def get_update_dict(dataset_id, variable, new_data):
    """Create a dictionary to pass to Visor's update_variables API."""
    update_dict = {
        "type": variable.type,
        "name": variable.name,
        "num_components": variable.num_components,
        "data": new_data
    }
    return update_dict

def update_variables(visor, dataset_id, part_variables, arrays):
    """Update the Visor dataset to display the new variable data."""
    update_dicts = []
    for variable in part_variables.variables:
        update_dicts.append(get_update_dict(dataset_id, variable, arrays.get(variable.name)))
    visor.update_variables(dataset_id, update_dicts)


In [ ]:
# Initialize the simulation, and run 60 steps, from which
# to create the VTK dataset and write to disk as VTKHDF.
sim = Example()
for _ in range(100):
    sim.step()

In [ ]:
# Create a plane, add the data from the initial sim pass as data arrays, and save to disk as vtkhdf.
create_dataset_and_write_to_disk(sim)

In [ ]:
# Instantiate VISOR, start it with the dataset, and get its variable info
# 
# Optional: update more than one VISOR instance e.g. to view more than one variable at once.
#           In this example: Visor instance 1 by default displays the density
#                            Visor instance 2 by default displays the pressure.
visor1, dataset_id1, variables1 = configure_visor(port=8081, variable_to_display="POINT::density::1")
visor2, dataset_id2, variables2 = configure_visor(port=8082, variable_to_display="POINT::pressure::1")

In [ ]:
IFrame(src="http://localhost:8081", width=800, height=400)

In [ ]:
IFrame(src="http://localhost:8082", width=800, height=400)

In [ ]:
for _ in range(200):
    sim.step()
    arrays = point_arrays(sim)
    update_variables(visor1, dataset_id1, variables1, arrays)
    update_variables(visor2, dataset_id2, variables2, arrays)
    await asyncio.sleep(0.4)